In [ ]:
import os
import gc
import tqdm
import numpy as np
import pandas as pd
from glob import glob
import concurrent.futures
import dask.dataframe as dd

code = 'B120'
only_mtm_col = True
files_folder_path = f"{code}/{code}_output/"
combine_folder_path = f"{code}/{code}_output_ParameterWise/"
os.makedirs(combine_folder_path, exist_ok=True)

EXT = "*.parquet"
all_files = [file for path, subdir, files in os.walk(files_folder_path) for file in glob(os.path.join(path, EXT))]
indices = sorted(set([f.split('\\')[-1].split(' ')[0] for f in all_files]))
os.makedirs(combine_folder_path, exist_ok=True)

print('Number of Files -', len(all_files))
print()

print('Indices -', indices)
print()

df = pd.read_parquet(max([(file, os.path.getsize(file)) for file in all_files], key=lambda x: x[-1])[0])
name_columns = [c for c in list(df.columns) if c.startswith('P_')]
pnl_columns = [c for c in list(df.columns) if c.endswith('PNL')]
print('Name cols-', list(map(lambda x: x.replace('P_', ''), name_columns)))
if only_mtm_col: print('\nPNL cols-', pnl_columns)

In [ ]:
def summarised_data(idx, name, df):
    
    summary_data = []
    for pnl_col in pnl_columns:
        df['CumulativePNL'] = df[pnl_col].cumsum()
        df['PeakPNL'] = df['CumulativePNL'].cummax()
        df['Drawdown'] = df['CumulativePNL'] - df['PeakPNL']
        df['DrawdownPct'] = df['Drawdown'] / df['PeakPNL'].replace(0, 1) * 100
        df['IsWin'] = (df[pnl_col] > 0).astype(int)

        total_trades = len(df)
        avg_pnl = df[pnl_col].mean()
        median_pnl = df[pnl_col].median()
        std_pnl = df[pnl_col].std()
        win_rate = (df[pnl_col] > 0).mean() * 100
        loss_rate = (df[pnl_col] < 0).mean() * 100

        avg_win = df[df[pnl_col] > 0][pnl_col].mean()
        avg_loss = df[df[pnl_col] < 0][pnl_col].mean()
        win_loss_ratio = avg_win / abs(avg_loss) if avg_loss != 0 else np.nan

        gross_profit = df[df[pnl_col] > 0][pnl_col].sum()
        gross_loss = df[df[pnl_col] < 0][pnl_col].sum()
        profit_factor = gross_profit / abs(gross_loss) if gross_loss != 0 else np.nan

        df['WinStreak'] = df['IsWin'] * (df['IsWin'].groupby((df['IsWin'] == 0).cumsum()).cumcount() + 1)
        df['LossStreak'] = (1 - df['IsWin']) * ((1 - df['IsWin']).groupby((df['IsWin'] == 1).cumsum()).cumcount() + 1)
        max_consec_win = df['WinStreak'].max()
        max_consec_loss = df['LossStreak'].max()

        max_drawdown = df['Drawdown'].min()
        max_drawdown_pct = df['DrawdownPct'].min()

        summary_data.append({
            "File Name": name,
            "PNL Column": pnl_col,
            "Total Trades": total_trades,
            "Average PNL": round(avg_pnl, 2),
            "Median PNL": round(median_pnl, 2),
            "Std Dev (Volatility)": round(std_pnl, 2),
            "Win Rate (%)": round(win_rate, 2),
            "Loss Rate (%)": round(loss_rate, 2),
            "Win/Loss Ratio": round(win_loss_ratio, 2) if not np.isnan(win_loss_ratio) else 'N/A',
            "Profit Factor": round(profit_factor, 2) if not np.isnan(profit_factor) else 'N/A',
            "Max Consecutive Wins": int(max_consec_win),
            "Max Consecutive Losses": int(max_consec_loss),
            "Max Drawdown": round(max_drawdown, 2),
            "Max Drawdown (%)": round(max_drawdown_pct, 2)
        })

    print(idx)
    return summary_data

In [ ]:
for index in indices:
    
    chunk_nos = sorted(set([k.split('-')[-1].split('.')[0] for k in glob(f'{files_folder_path}/{index}*')]))
    for chunk_no in chunk_nos:
        
        print(index, chunk_no)
        
        try:
            all_files = glob(f'{files_folder_path}/{index}*No-{chunk_no}.parquet')
            print(f'\nTotal file in {index} Chunks-{chunk_no} -', len(all_files))
            if len(all_files) == 0:continue

            print('Reading Chunks...')
            df = dd.read_parquet(all_files, columns=(name_columns+pnl_columns) if only_mtm_col else None)
            df = df.compute()
            print('Reading Complete...')

            os.makedirs(f"{combine_folder_path}{index}", exist_ok=True)
            print('Grouping Chunks...')
            grouped = df.groupby(name_columns)
            print('Grouping Complete...')

            with concurrent.futures.ThreadPoolExecutor(max_workers=os.cpu_count()*7) as executor:
                futures = [executor.submit(summarised_data, idx, name, df) for idx, (name, df) in enumerate(grouped)]
                
            results = []
            for f in concurrent.futures.as_completed(futures):
                results += f.result()

            del df
            gc.collect()
        except Exception as e:
            input(str(e))
    break

In [ ]:
len(results)